In [ ]:
import pandas as pd
from urllib.parse import urlparse

## Enrichissement par les projets NGI (Next Generation Internet)

Source : [NGI Discover Innovations](https://ngi.eu/discover-ngi-innovations/) -- export filtré sur la France.

**Objectif :** taguer les structures existantes de la base ESS numérique qui portent un projet NGI.

**Problème :** le CSV NGI contient des projets (titre, site web, description) mais aucun identifiant juridique (pas de SIREN). Le rapprochement avec la base SIRENE ne peut donc pas être automatique.

**Approche :**
1. Charger et filtrer les projets NGI (France uniquement)
2. Extraire le domaine des sites web pour faciliter le rapprochement
3. Produire une table de revue avec une colonne `siren` vide à remplir manuellement (via [annuaire-entreprises.data.gouv.fr](https://annuaire-entreprises.data.gouv.fr))
4. Une fois les SIREN renseignés, joindre avec la base ESS pour taguer les structures

In [4]:
# Chargement du CSV NGI (skiprows=1 pour ignorer la ligne "sep=,")
df_ngi = pd.read_csv('..\\..\\data\\raw\\csvExportNGI.csv', sep=',', skiprows=1)
print(f"{len(df_ngi)} projets NGI chargés")

# Filtrer les projets impliquant la France
df_ngi_fr = df_ngi[df_ngi['Country'].str.contains('france', case=False, na=False)].copy()
print(f"{len(df_ngi_fr)} projets impliquant la France")

46 projets NGI chargés
38 projets impliquant la France


## 2. Extraction des domaines et préparation de la table de revue

On extrait le domaine du site web de chaque projet pour faciliter la recherche manuelle du SIREN correspondant.

In [6]:
# Extraction du domaine depuis l'URL du projet
def extract_domain(url):
    try:
        return urlparse(str(url)).netloc.replace('www.', '')
    except:
        return ''

df_ngi_fr['domaine'] = df_ngi_fr['website'].apply(extract_domain)

# Table de revue : colonnes utiles + colonne siren vide à remplir manuellement
df_revue = df_ngi_fr[['title', 'domaine', 'website', 'Category', 'Project']].copy()
df_revue['siren'] = ''

print(f"{len(df_revue)} projets à rapprocher manuellement")
df_revue

38 projets à rapprocher manuellement


,title,domaine,website,Category,Project,siren
0,/e/,,https://e.foundation,decentralized solutions (including blockchain ...,ngi ledger,
1,AALLM (Adversarial audit of LLM-based search en…,,https://github.com/aiforensics/alexlab-interve...,measurement monitoring analysis and abuse hand...,ngi search,
3,ALIAS,,https://dapsi.ngi.eu/hall-of-fame/alias/,software engineering protocols interoperabilit...,dapsi,
4,ALLMA: an AI-based LLM for Android,,https://gitlab.e.foundation/e/os/open-source-l...,services and applications (for example email i...,ngi search,
5,CacheCash Experiment on EdgeNet,,https://ngiatlantic.eu/node/99,decentralized solutions (including blockchain ...,ngi atlantic,
6,CRYPTPAD,,https://dapsi.ngi.eu/hall-of-fame/cryptpad/,services and applications (for example email i...,dapsi,
7,D3ICA,,https://www.linkedin.com/showcase/d3ica/,decentralized solutions (including blockchain ...,ngi sargasso,
8,DAOstar,,https://ontochain.ngi.eu/content/daostar-seman...,network infrastructure (including routing peer...,ontochain,
9,Data space search engine (EDC),,https://git.startinblox.com/applications/ngi-s...,vertical use cases improving search and discov...,ngi search,
10,Datami,,https://gitlab.com/multi-coop/datami-project,services and applications (for example email i...,ngi search,


## 3. Export pour saisie manuelle des SIREN

Export en CSV de la table de revue. Pour chaque projet, rechercher le SIREN de la structure porteuse sur [annuaire-entreprises.data.gouv.fr](https://annuaire-entreprises.data.gouv.fr) et remplir la colonne `siren`.

In [7]:
output_path = '..\\..\\data\\processed\\ngi_revue_siren.csv'
df_revue.to_csv(output_path, index=False, sep=';', encoding='utf-8-sig')
print(f"Export : {output_path}")
print("→ Remplir la colonne 'siren' puis relancer la section 4")

Export : ..\..\data\processed\ngi_revue_siren.csv
→ Remplir la colonne 'siren' puis relancer la section 4


## 4. Jointure avec la base ESS (après saisie des SIREN)

Une fois le fichier `ngi_revue_siren.csv` complété, recharger ici et taguer les structures correspondantes dans la base ESS filtrée.

In [8]:
# Recharger le fichier complété
df_ngi_siren = pd.read_csv('..\\..\\data\\processed\\ngi_revue_siren.csv', sep=';', dtype={'siren': str})
df_ngi_siren = df_ngi_siren[df_ngi_siren['siren'].notna() & (df_ngi_siren['siren'] != '')]
print(f"{len(df_ngi_siren)} projets NGI avec SIREN renseigné")

# Charger la base ESS filtrée (sortie du notebook 01)
df_ess = pd.read_csv('..\\..\\data\\processed\\01_ess_numerique_export_liste.csv', dtype={'siren': str})

# Taguer les structures ESS portant un projet NGI
ngi_sirens = set(df_ngi_siren['siren'])
df_ess['tag_ngi'] = df_ess['siren'].isin(ngi_sirens)

n_tagged = df_ess['tag_ngi'].sum()
print(f"{n_tagged} structures ESS taguées 'projet NGI' sur {len(df_ess)}")
df_ess[df_ess['tag_ngi']]

0 projets NGI avec SIREN renseigné
0 structures ESS taguées 'projet NGI' sur 2060


,siren,denominationUniteLegale,nomUniteLegale,activitePrincipaleUniteLegale,trancheEffectifsUniteLegale,tag_ngi
